# Dialect ASR experiment on Colab

Fine-tunes Whisper small on ViMD speech from many provinces and compares the word error rate (WER) before and after.

**Before you start:** Runtime > Change runtime type > a GPU (T4 is enough).
ViMD is CC BY-NC-ND 4.0: use it for coursework and research only.

In [ ]:
!nvidia-smi -L
!df -h /content | tail -1   # free disk matters: see README "Disk is the limit"

## 1. Get the code and install

Upload `finetune.zip` (the zip of the `ai-service/speech/finetune` folder, files at the top level). No GitHub access is needed, so it works with a private repository.

When the cell asks, choose the zip file from your computer.

In [ ]:
import zipfile
from google.colab import files

# Leave the folder before deleting it, then remove older copies: Colab renames a re-uploaded
# file to "name (1).zip" and the old one would be read.
%cd /content
!rm -rf /content/finetune /content/*.zip

uploaded = files.upload()          # choose finetune-v2.zip (the newest zip)
zip_name = next(iter(uploaded))
zipfile.ZipFile(zip_name).extractall("finetune")
%cd /content/finetune
!ls
!grep -n "rev 2" finetune.py   # must print a line; if not, the old file is still there
!pip install -q -r requirements.txt

## 2. Choose provinces

Values: names without accents (`QuangNgai,KhanhHoa`), `coastal` (28 provinces), `region:Central`, `region:South`, `region:North`, `all`. They can be mixed.

- **First run: keep the 3-province smoke test** so a mistake shows up before a big download.
- **Real run:** train on a group that fits the disk, and evaluate on a wider one (test files are small).

In [ ]:
TRAIN_PROVINCES = "QuangNgai,KhanhHoa,BaRiaVungTau"   # smoke test; then e.g. "region:Central"
EVAL_PROVINCES = TRAIN_PROVINCES                       # then e.g. "coastal"
!python plan_download.py --provinces {TRAIN_PROVINCES}
!python plan_download.py --provinces {EVAL_PROVINCES}

## 3. Baseline: the model before fine-tuning

Run this first so there is something to compare with.

In [ ]:
!python eval_wer.py --model openai/whisper-small --provinces {EVAL_PROVINCES} --name baseline-whisper-small

Optional: the model the app uses today (Groq). Needs a free key from console.groq.com. Paste it when asked; **do not save the notebook with the key in it**. `--limit 5` keeps it to a few minutes.

In [ ]:
# import os, getpass; os.environ["GROQ_API_KEY"] = getpass.getpass("Groq key: ")
# !python eval_wer.py --model groq:whisper-large-v3-turbo --provinces {EVAL_PROVINCES} --limit 5 --name baseline-groq-turbo

## 4. Fine-tune

About 1000 steps takes roughly an hour on a T4 (an estimate, not measured). Lower `--max-steps` for a first try.

In [ ]:
!python finetune.py --train-provinces {TRAIN_PROVINCES} --max-steps 1000

## 5. Evaluate the fine-tuned model on the same test clips

In [ ]:
!python eval_wer.py --model outputs/whisper-small-vimd --provinces {EVAL_PROVINCES} --name finetuned-whisper-small

## 6. Compare and keep the results

The JSON files in `results/` hold the numbers. Download them (the model itself is large; save it to Drive only if you need it).

In [ ]:
import json, glob
for path in sorted(glob.glob("results/*-test.json")):
    r = json.load(open(path))
    print(f"{r['model']:<45} overall WER {r['overall_wer']:.1%}  ({r['clips']} clips)")
    for region, v in r["by_region"].items(): print(f"    region {region:<9}{v['wer']:.1%}  ({v['clips']} clips)")

In [ ]:
import os
from google.colab import files

if not (os.path.isdir("results") and os.listdir("results")):
    print("No results yet: an earlier step failed or did not run. Look at its error above.")
else:
    !zip -q -r results.zip results
    files.download("results.zip")

# Run 2: train on the whole Central region, measure on all coastal provinces

The first run used 3 provinces (about 740 training clips). This one trains on `region:Central` (19 provinces, about 4,700 clips) and measures on the 28 coastal provinces (about 920 test clips), so the numbers rest on far more clips. It uses new output names so run 1 is not overwritten.

Write down: the number of steps, how long training took, and the GPU. They go into the report.

In [ ]:
TRAIN_PROVINCES = "region:Central"
EVAL_PROVINCES = "coastal"
%cd /content/finetune
!df -h /content | tail -1          # free disk: the download below needs about 30 GB, and datasets keeps a second copy
!python plan_download.py --provinces {TRAIN_PROVINCES}
!python plan_download.py --provinces {EVAL_PROVINCES}

### Baseline of the stock model on the coastal test clips

In [ ]:
!python eval_wer.py --model openai/whisper-small --provinces {EVAL_PROVINCES} --name coastal-baseline-whisper-small

### Groq on the same clips (5 per province, about 8 minutes)

Paste the key when asked. Do not save the notebook with the key in it.

In [ ]:
import os, getpass
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq key: ")
!python eval_wer.py --model groq:whisper-large-v3-turbo --provinces {EVAL_PROVINCES} --limit 5 --name coastal-baseline-groq-turbo

### Fine-tune (about 1 to 1.5 hours on a T4, an estimate)

Keep this tab open. The model is saved under a new name.

In [ ]:
%%time
!python finetune.py --train-provinces {TRAIN_PROVINCES} --max-steps 1000 --output-dir /content/finetune/outputs/whisper-small-central

### Save the model to Drive right away (Colab deletes it when the session ends)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r /content/finetune/outputs/whisper-small-central /content/drive/MyDrive/whisper-small-central
!ls -la /content/drive/MyDrive/whisper-small-central

### How the training went

In [ ]:
import json, glob
for path in sorted(glob.glob("/content/finetune/outputs/whisper-small-central/checkpoint-*/trainer_state.json")):
    state = json.load(open(path))
    print(path.split("/")[-2], "| steps run:", state["global_step"], "of", state["max_steps"])
    for row in state["log_history"]:
        if "eval_wer" in row:
            print(f"   step {row['step']:>5}: validation WER {row['eval_wer']:.1%}")

### The fine-tuned model on the same coastal test clips

In [ ]:
!python eval_wer.py --model /content/finetune/outputs/whisper-small-central --provinces {EVAL_PROVINCES} --name coastal-finetuned-central

### Compare the three, and keep the results

In [ ]:
import json, glob
for path in sorted(glob.glob("results/coastal-*-test.json")):
    r = json.load(open(path))
    print(f"{path.split('/')[-1]:<48} overall WER {r['overall_wer']:.1%}  ({r['clips']} clips)")
    for region, v in r["by_region"].items(): print(f"    region {region:<9}{v['wer']:.1%}  ({v['clips']} clips)")
    for p in ("QuangNgai", "KhanhHoa", "BaRiaVungTau"):
        if p in r["by_province"]: print(f"    {p:<14}{r['by_province'][p]['wer']:.1%}  ({r['by_province'][p]['clips']} clips)")

In [ ]:
!zip -q -r results2.zip results
from google.colab import files
files.download("results2.zip")